# Korea Valuation 통합 조회 v5

**v5 변경점**: `merge_upside_rankings()` 신설 — 이미 추출된 순위표 DataFrame 2~3개(FCFF/RIM/Relative)를 ticker 기준으로 병합해 모형별 Upside + **동일가중 평균 Upside** 산출, 매출성장률_4Q/8Q·Moat 유지, 평균 Upside 내림차순 정렬. DB 재조회 없이 화면의 결과만 병합.

**v4 변경점**: 예측 DB와 실적 DB 간 **단위 불일치(10^k 배) 자동 감지·보정** — 종목별 (예측4Q합 ÷ 실적4Q합) 중앙값이 10^k(k≠0) 부근이면 예측값을 보정하고 로그로 알림. `unit_adjust` 파라미터로 수동 지정/해제 가능. `diagnose_growth()` 에 단위 비율 점검 추가.

**v3 변경점**: 매출 성장률을 종목별 루프 계산 → **예측 전량 로드 후 merge 방식**으로 재구성 (1회 로드·캐시 후 모든 순위표에 재사용), 단계별 로그(예측/실적 로드 건수, 매칭 수) 출력, NaN 원인 추적용 `diagnose_growth()` 신설.

**v2 변경점**: 매출 예측 DB(`korea_revenue_forecast_result`) 연동 — 각 순위표의 Upside 칼럼 옆에 **12개월(4분기)·24개월(8분기) 매출 성장률 예측** 칼럼 추가, 순수 매출 성장률 조회 함수 `get_revenue_growth()` 신설.

FCFF DCF · RIM · Relative Valuation 3개 모형의 DB 저장 결과를 통합 조회하고
upside 순위로 추출 → 엑셀(`C:\valuation results`) 저장하는 노트북.

| 함수 | 설명 |
|---|---|
| `get_valuation_dates()` | 모형별 valuation 수행 날짜 + 종목 수 조회 |
| `get_fcff_ranking()` | FCFF upside 순위 추출 |
| `get_rim_ranking()` | RIM upside 순위 추출 |
| `get_relative_ranking()` | 상대가치 upside 순위 추출 (PER/PBR/PSR 선택) |
| `get_combined_ranking()` | 3개 모형 통합 비교 순위 (DB 재조회) |
| `merge_upside_rankings()` | 추출된 순위표 2~3개 병합 → 동일가중 평균 upside 순위 (★v5) |
| `get_revenue_growth()` | 매출 성장률(4Q/8Q) 단독 조회 (★v2) |
| `build_growth_table()` | 전 종목 성장률 테이블 생성·캐시 (★v3) |
| `diagnose_growth()` | 특정 종목 성장률 NaN 원인 진단 (★v3) |

공통 파라미터:
- `n=50` : 상위 N개 (rank_range 미지정 시)
- `rank_range=(100, 150)` : 순위 구간 출력 (지정 시 n 무시)
- `dates=["2026-06-01", "2026-06-02"]` : 평가일 리스트. `None` → 최신 평가일 자동.
  여러 날 중복 종목은 **최신 측정일 기준** 1행만 사용.
- `save=True` : 엑셀 저장 여부 (기본 True). `file_format="csv"` 도 가능.
- `max_upside=None` : 데이터 품질용 upside 상한 필터 (예: 300 → +300% 초과 제외)
- `growth_model="Ensemble"` : 매출 성장률 산출 모델. `"SARIMA"`/`"ETS"`/`"Theta"` 지정 가능, `None` → 성장률 칼럼 생략 (★v2)

**매출 성장률 정의 (★v2)**
- `매출성장률_4Q(%)` = Σ(예측 1~4분기 매출) ÷ Σ(예측 직전 4개 실적 분기 매출) − 1 → 향후 12개월 vs 직전 12개월
- `매출성장률_8Q(%)` = Σ(예측 1~8분기 매출) ÷ Σ(예측 직전 8개 실적 분기 매출) − 1 → 향후 24개월 vs 직전 24개월
- 종목별로 **가장 최근 예측 실행일(created_at)** 버전 사용. 실적/예측 분기 수 부족 시 NaN.


In [1]:
# ── Cell 1 · 경로 자동 감지 (기존 valuation 노트북과 동일) ──────
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 환경에 맞게 수정하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [2]:
# ── Cell 2 · Import & 설정 상수 ─────────────────────────────────
from datetime import datetime
from typing import Optional, List, Tuple, Union, Dict

import numpy as np
import pandas as pd
from sqlalchemy import text
from IPython.display import display

from DATA.config import get_db_info, get_engine
from DATA.korea_valuation_helpers import to_price_ticker

def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ══════════════════════════════════════════════════════════════
#  설정 — 여기만 수정하세요
# ══════════════════════════════════════════════════════════════

# 3개 valuation 결과 테이블 (각 노트북의 TABLE_RESULT 와 동일)
TABLE_FCFF = "korea_fcff_dcf_valuation_v7"   # FCFF DCF v7
TABLE_RIM  = "korea_rim_valuation"            # RIM
TABLE_REL  = "korea_relative_valuation"       # Relative (PBR/PSR/PER)

# ★v2: 매출 예측 결과 (long: date·ticker·indicator·value·created_at) / 실적 원천
TABLE_FORECAST    = "korea_revenue_forecast_result"
TABLE_FS          = "korea_fs_data_from_DG"
REVENUE_ITEM_CODE = "M000904001"   # 매출액(천원)
GROWTH_MODELS     = {"ENSEMBLE": "Ensemble", "SARIMA": "SARIMA",
                     "ETS": "ETS", "THETA": "Theta"}

# 엑셀 저장 폴더
OUTPUT_DIR = r"C:\valuation results"

# 상대가치 지표명 ↔ DB 컬럼 suffix 매핑
#  ※ 상대가치 노트북은 PER/PBR/"PSR(주가매출비율)" 3종을 저장합니다.
#    PCR(주가현금흐름비율)은 DB에 없으므로 PSR 이 그 자리를 대신합니다.
REL_METRICS_ALL = ["PER", "PBR", "PSR"]

db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

print(f"[설정] FCFF={TABLE_FCFF}  RIM={TABLE_RIM}  REL={TABLE_REL}")
print(f"[설정] 저장 폴더 = {OUTPUT_DIR}")


[22:42:18][DB] 연결 성공 host=192.168.0.230 port=3307
[설정] FCFF=korea_fcff_dcf_valuation_v7  RIM=korea_rim_valuation  REL=korea_relative_valuation
[설정] 저장 폴더 = C:\valuation results


In [3]:
# ── Cell 3 · 종목명/섹터 매핑 (FDR StockListing, 실패해도 진행) ──
NAME_LOOKUP: Dict[str, Dict[str, str]] = {}
try:
    import FinanceDataReader as fdr
    _listing = fdr.StockListing("KRX")
    _code_col = next((c for c in ["Code", "Symbol", "code", "ticker"]
                      if c in _listing.columns), None)
    _name_col = next((c for c in ["Name", "name", "기업명"]
                      if c in _listing.columns), None)
    _sec_col  = next((c for c in ["Sector", "Industry", "업종", "sector", "industry"]
                      if c in _listing.columns), None)
    if _code_col and _name_col:
        for _, row in _listing.iterrows():
            code = str(row[_code_col]).strip().zfill(6)
            NAME_LOOKUP[code] = {
                "name":   str(row[_name_col]) if pd.notna(row[_name_col]) else "",
                "sector": str(row[_sec_col])  if _sec_col and pd.notna(row.get(_sec_col)) else "",
            }
        log("NAME", f"FDR StockListing 로드 → {len(NAME_LOOKUP):,}개 종목명 매핑")
    else:
        log("NAME", f"[WARN] FDR 컬럼 불일치 → 종목명 공란 (cols={list(_listing.columns)})")
except Exception as e:
    log("NAME", f"[WARN] FDR StockListing 실패: {e}")
    log("NAME", "       → 종목명 공란으로 진행합니다. "
                "pip install -U finance-datareader 후 재실행하거나 "
                "set_names_from_df() 로 수동 매핑하세요.")


def get_name_sector(ticker: str) -> Dict[str, str]:
    """'A005930' / '005930' → {'name': '삼성전자', 'sector': ...}"""
    try:
        code = to_price_ticker(ticker)
    except Exception:
        code = str(ticker).lstrip("A").zfill(6)
    return NAME_LOOKUP.get(code, {"name": "", "sector": ""})


def set_names_from_df(df: pd.DataFrame,
                      ticker_col: str = "ticker",
                      name_col: str = "name") -> int:
    """FDR 불가 환경용: 임의 DataFrame(CSV 등)으로 종목명 매핑 주입.   ★v3

    예) names = pd.read_csv("krx_names.csv"); set_names_from_df(names, "Code", "Name")
    """
    n = 0
    for _, row in df[[ticker_col, name_col]].dropna().iterrows():
        code = str(row[ticker_col]).strip().upper().lstrip("A").zfill(6)
        NAME_LOOKUP.setdefault(code, {"name": "", "sector": ""})
        NAME_LOOKUP[code]["name"] = str(row[name_col])
        n += 1
    log("NAME", f"수동 매핑 {n}건 반영 (총 {len(NAME_LOOKUP):,}건)")
    return n


[22:42:22][NAME] [WARN] FDR StockListing 실패: Expecting value: line 1 column 1 (char 0)
[22:42:22][NAME]        → 종목명 공란으로 진행합니다. pip install -U finance-datareader 후 재실행하거나 set_names_from_df() 로 수동 매핑하세요.


In [4]:
# ── Cell 4 · 공통 유틸 (날짜 정규화 / 순위 슬라이스 / 파일 저장) ──

_MODEL_TABLE = {"FCFF": TABLE_FCFF, "RIM": TABLE_RIM, "Relative": TABLE_REL}


def get_valuation_dates(model: str = "all") -> pd.DataFrame:
    """valuation 수행 날짜 조회.

    Parameters
    ----------
    model : "fcff" | "rim" | "relative" | "all"

    Returns
    -------
    DataFrame [model, run_date, n_tickers, n_rows]  (최신순)
    """
    key = model.strip().lower()
    targets = (_MODEL_TABLE.items() if key == "all"
               else [(m, t) for m, t in _MODEL_TABLE.items() if m.lower() == key])
    if not targets:
        raise ValueError(f"model='{model}' 인식 불가. 'fcff'/'rim'/'relative'/'all' 중 선택.")

    frames = []
    for mname, tbl in targets:
        sql = text(f"""
            SELECT '{mname}' AS model, `date` AS run_date,
                   COUNT(DISTINCT ticker) AS n_tickers, COUNT(*) AS n_rows
            FROM `{tbl}`
            GROUP BY `date`
            ORDER BY `date` DESC
        """)
        try:
            df = pd.read_sql(sql, engine)
            frames.append(df)
        except Exception as e:
            log("DATES", f"[WARN] {mname}({tbl}) 조회 실패: {e}")
    if not frames:
        return pd.DataFrame(columns=["model", "run_date", "n_tickers", "n_rows"])
    out = pd.concat(frames, ignore_index=True)
    out["run_date"] = pd.to_datetime(out["run_date"]).dt.strftime("%Y-%m-%d")
    return out.sort_values(["model", "run_date"],
                           ascending=[True, False]).reset_index(drop=True)


def _resolve_dates(dates, table: str, model_name: str) -> List[str]:
    """입력 dates(None/str/list) → 해당 테이블에 실제 존재하는 날짜 리스트.

    - None / [] → 최신 평가일 1일 자동
    - 존재하지 않는 날짜는 경고 후 제외
    """
    avail = pd.read_sql(
        text(f"SELECT DISTINCT `date` FROM `{table}` ORDER BY `date` DESC"), engine)
    if avail.empty:
        raise ValueError(f"[{model_name}] {table} 에 저장된 결과가 없습니다.")
    avail_set = set(pd.to_datetime(avail["date"]).dt.strftime("%Y-%m-%d"))

    if dates is None or (isinstance(dates, (list, tuple)) and len(dates) == 0):
        latest = max(avail_set)
        log(model_name, f"날짜 미지정 → 최신 평가일 사용: {latest}")
        return [latest]

    if isinstance(dates, str):
        dates = [dates]
    norm = [pd.to_datetime(d).strftime("%Y-%m-%d") for d in dates]
    missing = sorted(set(norm) - avail_set)
    if missing:
        log(model_name, f"⚠️  측정 기록 없는 날짜 (무시): {missing}")
    valid = sorted(set(norm) & avail_set)
    if not valid:
        raise ValueError(f"[{model_name}] 입력한 날짜에 측정 기록이 없습니다. "
                         f"get_valuation_dates('{model_name.lower()}') 로 확인하세요.")
    return valid


def _slice_rank(df: pd.DataFrame, n: int,
                rank_range: Optional[Tuple[int, int]]) -> pd.DataFrame:
    """upside 내림차순 정렬 후 rank 부여 → 상위 n 또는 rank_range 구간."""
    df = df.reset_index(drop=True)
    df.insert(0, "rank", df.index + 1)
    if rank_range is not None:
        lo, hi = int(rank_range[0]), int(rank_range[1])
        if lo > hi:
            lo, hi = hi, lo
        out = df[(df["rank"] >= lo) & (df["rank"] <= hi)]
        if out.empty:
            log("RANK", f"⚠️  rank_range=({lo},{hi}) 구간에 종목 없음 "
                        f"(전체 {len(df)}개)")
        return out
    return df.head(int(n))


def _save_output(df: pd.DataFrame, method: str, run_dates: List[str],
                 save: bool = True, file_format: str = "xlsx") -> Optional[str]:
    """OUTPUT_DIR 에 {Method}_valuation_{수행날짜}_{출력날짜}.xlsx 형식으로 저장."""
    if not save:
        return None
    if df.empty:
        log("SAVE", "결과가 비어 있어 저장 생략")
        return None
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    run_tag = (run_dates[0] if len(run_dates) == 1
               else f"{min(run_dates)}_to_{max(run_dates)}")
    today = datetime.now().strftime("%Y-%m-%d")
    ext = "csv" if str(file_format).lower() == "csv" else "xlsx"
    fname = f"{method}_valuation_{run_tag}_{today}.{ext}"
    path = os.path.join(OUTPUT_DIR, fname)
    if ext == "csv":
        df.to_csv(path, index=False, encoding="utf-8-sig")
    else:
        df.to_excel(path, index=False)
    log("SAVE", f"저장 완료: {path}  ({len(df)}행)")
    return path


def _attach_name(df: pd.DataFrame, after_col: str = "ticker") -> pd.DataFrame:
    """ticker 다음 위치에 종목명 컬럼 삽입."""
    names = df["ticker"].map(lambda t: get_name_sector(t)["name"])
    pos = df.columns.get_loc(after_col) + 1
    df.insert(pos, "종목명", names)
    return df


In [5]:
# ── Cell 5 (★v3) · 매출 성장률: 예측 전량 로드 → merge 방식 ────
#
#  구조 (v2 의 종목별 루프 → v3 벡터화 + 캐시)
#    1) load_revenue_forecast_all : 모델별 '종목별 최신 created_at' 예측 전량 로드
#    2) load_revenue_actual_all   : 해당 종목들의 실적 매출 일괄 로드
#    3) build_growth_table        : groupby 벡터 연산으로 성장률 테이블 생성 → 캐시
#    4) _merge_growth             : 순위표와 ticker merge (매칭 건수 로그)
#    5) diagnose_growth           : 특정 종목 NaN 원인 단계별 출력
#
#  정의:  매출성장률_4Q = Σ예측(1~4분기) / Σ실적(예측 직전 4분기) − 1
#         매출성장률_8Q = Σ예측(1~8분기) / Σ실적(예측 직전 8분기) − 1

import re as _re

_GROWTH_CACHE: Dict[str, pd.DataFrame] = {}
_GROWTH_COLS = ["매출성장률_4Q(%)", "매출성장률_8Q(%)"]


def _norm_tk(t) -> str:
    """'005930' / 'a005930 ' / 5930 → 'A005930' (merge 키 통일)."""
    s = str(t).strip().upper()
    body = s[1:] if s.startswith("A") else s
    return "A" + body.zfill(6)


def _canon_model(model: str) -> str:
    key = str(model).strip().upper()
    if key not in GROWTH_MODELS:
        raise ValueError(f"model='{model}' 인식 불가. "
                         f"사용 가능: {list(GROWTH_MODELS.values())}")
    return GROWTH_MODELS[key]


def load_revenue_forecast_all(model: str = "Ensemble",
                              verbose: bool = True) -> pd.DataFrame:
    """예측 DB에서 해당 모델의 '종목별 최신 created_at 버전' 예측을 전량 로드 (long)."""
    ind = _canon_model(model)
    sql = text(f"""
        SELECT f.ticker, f.`date` AS fc_date, f.value AS fc_value, f.created_at
        FROM `{TABLE_FORECAST}` f
        JOIN (
            SELECT ticker, MAX(created_at) AS ca
            FROM `{TABLE_FORECAST}`
            WHERE indicator = :ind
            GROUP BY ticker
        ) x ON f.ticker = x.ticker AND f.created_at = x.ca
        WHERE f.indicator = :ind
          AND f.value IS NOT NULL
        ORDER BY f.ticker, f.`date`
    """)
    fc = pd.read_sql(sql, engine, params={"ind": ind})
    if fc.empty:
        log("GROWTH", f"⚠️ [{ind}] 예측 데이터 0행 — indicator 표기를 확인하세요: "
                      f"SELECT DISTINCT indicator FROM {TABLE_FORECAST}")
        return fc
    fc["ticker"]    = fc["ticker"].map(_norm_tk)
    fc["fc_period"] = pd.PeriodIndex(pd.to_datetime(fc["fc_date"]), freq="Q")
    if verbose:
        log("GROWTH", f"[{ind}] 예측 로드 {len(fc):,}행 / {fc['ticker'].nunique():,}종목 | "
                      f"실행일 {fc['created_at'].min()}~{fc['created_at'].max()} | "
                      f"예측구간 {fc['fc_period'].min()}~{fc['fc_period'].max()}")
    return fc


def load_revenue_actual_all(tickers, years: int = 5,
                            verbose: bool = True) -> pd.DataFrame:
    """실적 분기 매출 일괄 로드 (동일 분기 중복 → 마지막 값, forecast 노트북과 동일 규칙)."""
    tks = sorted({_norm_tk(t) for t in tickers})
    tks = [t for t in tks if _re.fullmatch(r"A\d{6}", t)]
    if not tks:
        return pd.DataFrame()
    in_list = ", ".join(f"'{t}'" for t in tks)
    cutoff = (pd.Timestamp.today() - pd.DateOffset(years=years)).strftime("%Y-%m-%d")
    sql = text(f"""
        SELECT ticker, `date` AS act_date, value AS act_value
        FROM `{TABLE_FS}`
        WHERE item_code = :item
          AND value IS NOT NULL
          AND `date` >= :cutoff
          AND ticker IN ({in_list})
        ORDER BY ticker, `date`
    """)
    act = pd.read_sql(sql, engine, params={"item": REVENUE_ITEM_CODE, "cutoff": cutoff})
    if act.empty:
        log("GROWTH", f"⚠️ 실적 매출 0행 — {TABLE_FS} 에 item_code='{REVENUE_ITEM_CODE}' "
                      f"(매출액) 데이터가 있는지 확인하세요.")
        return act
    act["ticker"]     = act["ticker"].map(_norm_tk)
    act["act_period"] = pd.PeriodIndex(pd.to_datetime(act["act_date"]), freq="Q")
    act = (act.sort_values(["ticker", "act_period", "act_date"])
              .groupby(["ticker", "act_period"], as_index=False).last())
    if verbose:
        log("GROWTH", f"실적 로드 {len(act):,}행 / {act['ticker'].nunique():,}종목 "
                      f"(최근 {years}년)")
    return act


def build_growth_table(model: str = "Ensemble",
                       refresh: bool = False,
                       verbose: bool = True,
                       unit_adjust: Union[str, float, None] = "auto") -> pd.DataFrame:
    """전 종목 매출 성장률 테이블 생성 + 캐시.   ★v3 핵심 / ★v4 단위 보정

    한 번 만들면 _GROWTH_CACHE 에 저장되어 같은 세션의 모든 순위표가 재사용.
    예측 재실행 후에는 refresh=True 로 갱신.

    unit_adjust : ★v4 예측↔실적 단위 불일치 처리
        "auto"(기본) → (예측4Q합/실적4Q합) 중앙값이 10^k (k≠0) 부근이면
                       예측값에 10^(-k) 를 곱해 자동 보정 (로그로 알림)
        1.0 / None   → 보정 안 함 (원본 그대로)
        float        → 예측값에 해당 배수를 직접 곱함 (예: 0.001)
    """
    ind = _canon_model(model)
    cache_key = f"{ind}|{unit_adjust}"
    if not refresh and cache_key in _GROWTH_CACHE:
        return _GROWTH_CACHE[cache_key]

    fc = load_revenue_forecast_all(ind, verbose=verbose)
    if fc.empty:
        return pd.DataFrame()
    act = load_revenue_actual_all(fc["ticker"].unique(), verbose=verbose)

    # ── 예측측: 종목별 앞 4Q / 8Q 합 ──
    fc = fc.sort_values(["ticker", "fc_period"]).reset_index(drop=True)
    fc["q_idx"] = fc.groupby("ticker").cumcount() + 1
    base = fc.groupby("ticker").agg(
        first_fc=("fc_period", "first"),
        created_at=("created_at", "first"),
        fc_n=("fc_value", "size"))
    f4 = fc.loc[fc["q_idx"] <= 4].groupby("ticker")["fc_value"].sum().rename("f4_sum")
    f8 = fc.loc[fc["q_idx"] <= 8].groupby("ticker")["fc_value"].sum().rename("f8_sum")
    tbl = base.join([f4, f8])

    # ── 실적측: 예측 시작분기 '이전' 구간의 뒤 4Q / 8Q 합 ──
    if not act.empty:
        act = act.merge(tbl[["first_fc"]], left_on="ticker",
                        right_index=True, how="inner")
        act = act[act["act_period"] < act["first_fc"]]
        act = act.sort_values(["ticker", "act_period"])
        act["r_idx"] = act.groupby("ticker").cumcount(ascending=False) + 1
        a4 = (act.loc[act["r_idx"] <= 4].groupby("ticker")["act_value"]
                 .agg(a4_sum="sum", a4_n="size"))
        a8 = (act.loc[act["r_idx"] <= 8].groupby("ticker")["act_value"]
                 .agg(a8_sum="sum", a8_n="size"))
        tbl = tbl.join([a4, a8])
    for c in ["a4_sum", "a4_n", "a8_sum", "a8_n"]:
        if c not in tbl.columns:
            tbl[c] = np.nan
    tbl[["a4_n", "a8_n"]] = tbl[["a4_n", "a8_n"]].fillna(0)

    # ── ★v4 단위 정합성 점검·보정 ──────────────────────────────
    #   예측과 실적은 같은 매출 시계열이므로 (예측4Q합/실적4Q합) ≈ 1±성장률.
    #   중앙값이 10^k (k≠0) 부근이면 단위 불일치(천원↔원 등)로 판단.
    scale = 1.0
    chk = tbl[(tbl["a4_n"] >= 4) & (tbl["a4_sum"] > 0) & tbl["f4_sum"].notna()]
    med_ratio = float((chk["f4_sum"] / chk["a4_sum"]).median()) if len(chk) else np.nan

    if unit_adjust is None:
        scale = 1.0
    elif isinstance(unit_adjust, (int, float)):
        scale = float(unit_adjust)
        if scale != 1.0:
            log("GROWTH", f"⚠️ [{ind}] 예측값 수동 단위보정 ×{scale:g} 적용")
    elif str(unit_adjust).lower() == "auto" and np.isfinite(med_ratio) and med_ratio > 0:
        k = int(np.round(np.log10(med_ratio)))
        if k != 0 and abs(k) <= 6 and abs(np.log10(med_ratio) - k) < 0.35:
            scale = 10.0 ** (-k)
            log("GROWTH", f"⚠️ [{ind}] 단위 불일치 감지: 예측합/실적합 중앙값 "
                          f"≈ {med_ratio:,.0f} (10^{k}) → 예측값에 ×{scale:g} 자동 보정")
            log("GROWTH", f"   원천 단위를 점검하세요: {TABLE_FORECAST}.value vs "
                          f"{TABLE_FS}(item={REVENUE_ITEM_CODE}). "
                          f"보정 끄려면 unit_adjust=1.0")
        elif verbose:
            log("GROWTH", f"[{ind}] 단위 점검 OK (예측합/실적합 중앙값 {med_ratio:.3f})")

    if scale != 1.0:
        tbl["f4_sum"] = tbl["f4_sum"] * scale
        tbl["f8_sum"] = tbl["f8_sum"] * scale

    def _g(f_sum, a_sum, a_n, need_a, fc_n, need_f):
        ok = (fc_n >= need_f) & (a_n >= need_a) & (a_sum > 0)
        return np.where(ok, (f_sum / a_sum - 1.0) * 100.0, np.nan)

    tbl["매출성장률_4Q(%)"] = np.round(
        _g(tbl["f4_sum"], tbl["a4_sum"], tbl["a4_n"], 4, tbl["fc_n"], 4), 1)
    tbl["매출성장률_8Q(%)"] = np.round(
        _g(tbl["f8_sum"], tbl["a8_sum"], tbl["a8_n"], 8, tbl["fc_n"], 8), 1)

    tbl = tbl.reset_index()
    tbl["모델"]       = ind
    tbl["예측실행일"]  = pd.to_datetime(tbl["created_at"]).dt.strftime("%Y-%m-%d")
    tbl["예측시작분기"] = tbl["first_fc"].astype(str)
    _uk = 1_000 / 1e8   # 천원 → 억원
    tbl["직전4Q매출(억원)"] = (tbl["a4_sum"] * _uk).round(0)
    tbl["향후4Q매출(억원)"] = (tbl["f4_sum"] * _uk).round(0)
    tbl["직전8Q매출(억원)"] = (tbl["a8_sum"] * _uk).round(0)
    tbl["향후8Q매출(억원)"] = (tbl["f8_sum"] * _uk).round(0)

    n4 = int(tbl["매출성장률_4Q(%)"].notna().sum())
    n8 = int(tbl["매출성장률_8Q(%)"].notna().sum())
    if verbose:
        log("GROWTH", f"[{ind}] 성장률 테이블 {len(tbl):,}종목 "
                      f"(4Q 유효 {n4:,} / 8Q 유효 {n8:,})")
    if n4 == 0:
        log("GROWTH", "⚠️ 4Q 성장률이 전부 NaN 입니다. "
                      "diagnose_growth('005930') 으로 원인을 확인하세요. "
                      "(예측분기수 부족 / 예측 이전 실적분기 부족 / 실적 0행 중 하나)")

    _GROWTH_CACHE[cache_key] = tbl
    return tbl


def get_revenue_growth(tickers: Union[None, str, List[str]] = None,
                       model: str = "Ensemble",
                       detail: bool = False,
                       refresh: bool = False,
                       unit_adjust: Union[str, float, None] = "auto") -> pd.DataFrame:
    """매출 성장률 단독 조회 (캐시된 성장률 테이블에서 추출).

    tickers : None → 전체 / '005930' / ['A005930', ...]
    model   : "Ensemble"(기본) | "SARIMA" | "ETS" | "Theta"
    detail  : True → 직전/향후 매출 절대금액(억원)·분기수 포함
    refresh : True → DB에서 다시 로드 (예측 재실행 직후)
    """
    tbl = build_growth_table(model, refresh=refresh, unit_adjust=unit_adjust)
    if tbl.empty:
        return tbl
    if tickers is not None:
        if isinstance(tickers, str):
            tickers = [tickers]
        keys = {_norm_tk(t) for t in tickers}
        tbl = tbl[tbl["ticker"].isin(keys)]

    cols = ["ticker", "예측실행일", "예측시작분기", "모델"] + _GROWTH_COLS
    if detail:
        cols += ["직전4Q매출(억원)", "향후4Q매출(억원)",
                 "직전8Q매출(억원)", "향후8Q매출(억원)", "fc_n", "a4_n", "a8_n"]
    out = tbl[cols].rename(columns={"fc_n": "예측분기수",
                                    "a4_n": "실적4Q수", "a8_n": "실적8Q수"})
    out = _attach_name(out.copy())
    return out.reset_index(drop=True)


def _merge_growth(rank_df: pd.DataFrame,
                  growth_model: Optional[str]) -> pd.DataFrame:
    """순위표의 Upside 칼럼 옆에 성장률 merge.   ★v3: 캐시 테이블 사용 + 매칭 로그"""
    if growth_model is None or rank_df.empty or "ticker" not in rank_df.columns:
        return rank_df
    try:
        tbl = build_growth_table(growth_model, verbose=False)
    except Exception as e:
        log("GROWTH", f"[WARN] 성장률 테이블 생성 실패 → 칼럼 생략: {e}")
        return rank_df
    if tbl.empty:
        log("GROWTH", "[WARN] 성장률 데이터 없음 → 칼럼 생략")
        return rank_df

    left = rank_df.copy()
    left["_tk_norm"] = left["ticker"].map(_norm_tk)
    right = tbl[["ticker"] + _GROWTH_COLS].rename(columns={"ticker": "_tk_norm"})
    merged = left.merge(right, on="_tk_norm", how="left").drop(columns=["_tk_norm"])

    matched = int(merged[_GROWTH_COLS[0]].notna().sum()
                  + (merged[_GROWTH_COLS[0]].isna()
                     & merged[_GROWTH_COLS[1]].notna()).sum())
    n_key = int(left["_tk_norm"].isin(set(right["_tk_norm"])).sum())
    log("GROWTH", f"순위표 {len(left)}종목 중 예측DB 매칭 {n_key}종목 / "
                  f"성장률 유효 {matched}종목 (model={_canon_model(growth_model)})")

    # 첫 Upside 칼럼 바로 뒤로 이동
    up_cols = [c for c in merged.columns if str(c).startswith("Upside")]
    if up_cols:
        anchor = merged.columns.get_loc(up_cols[0])
        order = [c for c in merged.columns if c not in _GROWTH_COLS]
        order = order[:anchor + 1] + _GROWTH_COLS + order[anchor + 1:]
        merged = merged[order]
    return merged


def diagnose_growth(ticker, model: str = "Ensemble") -> None:
    """특정 종목의 성장률 NaN 원인 단계별 진단.   ★v3

    예측 rows / 실적 rows / 정렬 구간 / 합계 / 판정 조건을 모두 출력합니다.
    """
    ind = _canon_model(model)
    tk = _norm_tk(ticker)
    print("=" * 78)
    print(f" 성장률 진단  |  ticker={tk}  model={ind}")
    print("=" * 78)

    # 1) 예측
    fc = pd.read_sql(text(f"""
        SELECT f.`date` AS fc_date, f.value AS fc_value, f.created_at
        FROM `{TABLE_FORECAST}` f
        JOIN (SELECT MAX(created_at) AS ca FROM `{TABLE_FORECAST}`
              WHERE indicator=:ind AND ticker=:tk) x
          ON f.created_at = x.ca
        WHERE f.indicator=:ind AND f.ticker=:tk AND f.value IS NOT NULL
        ORDER BY f.`date`
    """), engine, params={"ind": ind, "tk": tk})
    if fc.empty:
        # 형식/표기 문제인지 확인
        chk = pd.read_sql(text(f"""
            SELECT indicator, COUNT(*) AS n FROM `{TABLE_FORECAST}`
            WHERE ticker = :tk GROUP BY indicator"""), engine, params={"tk": tk})
        print(f"[1] 예측 데이터: 0행  ← ★원인")
        if chk.empty:
            print(f"    → {TABLE_FORECAST} 에 ticker='{tk}' 자체가 없습니다. "
                  f"ticker 형식(A접두사)을 확인하세요.")
            samp = pd.read_sql(text(
                f"SELECT DISTINCT ticker FROM `{TABLE_FORECAST}` LIMIT 5"), engine)
            print(f"    → 예측 테이블 ticker 예시: {list(samp['ticker'])}")
        else:
            print(f"    → 해당 종목의 indicator 분포:\n{chk.to_string(index=False)}")
            print(f"    → indicator='{ind}' 표기가 위 목록과 일치하는지 확인하세요.")
        return
    fc["period"] = pd.PeriodIndex(pd.to_datetime(fc["fc_date"]), freq="Q")
    first_fc = fc["period"].iloc[0]
    print(f"[1] 예측 데이터: {len(fc)}행  (실행일={fc['created_at'].iloc[0]}, "
          f"예측구간 {fc['period'].iloc[0]}~{fc['period'].iloc[-1]})")
    print(fc[["period", "fc_value"]].to_string(
        index=False, float_format=lambda x: f"{x:,.0f}"))

    # 2) 실적
    act = pd.read_sql(text(f"""
        SELECT `date` AS act_date, value AS act_value
        FROM `{TABLE_FS}`
        WHERE ticker=:tk AND item_code=:item AND value IS NOT NULL
        ORDER BY `date`
    """), engine, params={"tk": tk, "item": REVENUE_ITEM_CODE})
    if act.empty:
        print(f"\n[2] 실적 매출: 0행  ← ★원인")
        print(f"    → {TABLE_FS} / item_code='{REVENUE_ITEM_CODE}' 조회 결과 없음.")
        chk = pd.read_sql(text(f"""
            SELECT item_code, COUNT(*) AS n FROM `{TABLE_FS}`
            WHERE ticker=:tk GROUP BY item_code LIMIT 10"""),
            engine, params={"tk": tk})
        if not chk.empty:
            print(f"    → 해당 종목 보유 item_code 예시:\n{chk.to_string(index=False)}")
        return
    act["period"] = pd.PeriodIndex(pd.to_datetime(act["act_date"]), freq="Q")
    act = (act.sort_values(["period", "act_date"])
              .groupby("period", as_index=False).last())
    before = act[act["period"] < first_fc]
    print(f"\n[2] 실적 매출: 총 {len(act)}분기, 예측시작({first_fc}) 이전 {len(before)}분기")
    print(before.tail(10)[["period", "act_value"]].to_string(
        index=False, float_format=lambda x: f"{x:,.0f}"))

    # 3) ★v4 단위 정합성 점검 (마지막 실적분기 vs 첫 예측분기)
    last_act = float(before["act_value"].iloc[-1]) if len(before) else np.nan
    first_val = float(fc["fc_value"].iloc[0])
    if np.isfinite(last_act) and last_act > 0:
        r = first_val / last_act
        print(f"\n[3] 단위 점검: 첫 예측분기 {first_val:,.0f} / "
              f"마지막 실적분기 {last_act:,.0f} = 비율 {r:,.2f}")
        if r > 30 or r < 1/30:
            k = int(np.round(np.log10(r)))
            print(f"    ⚠️ 비율이 10^{k} 수준 → 예측↔실적 단위 불일치 가능성 큼. "
                  f"build_growth_table 의 unit_adjust='auto' 가 자동 보정합니다.")
        else:
            print(f"    ✅ 단위 정합 (성장률 범위 내)")

    # 4) 판정 (unit_adjust='auto' 와 동일한 보정 적용)
    _scale = 1.0
    if np.isfinite(last_act) and last_act > 0:
        _r = first_val / last_act
        _k = int(np.round(np.log10(_r))) if _r > 0 else 0
        if _k != 0 and abs(_k) <= 6 and abs(np.log10(_r) - _k) < 0.35:
            _scale = 10.0 ** (-_k)
    print(f"\n[4] 판정" + (f"  (예측값 ×{_scale:g} 보정 적용)" if _scale != 1.0 else ""))
    for n, label in [(4, "4Q(12개월)"), (8, "8Q(24개월)")]:
        ok_f = len(fc) >= n
        ok_a = len(before) >= n
        f_sum = fc["fc_value"].iloc[:n].sum() * _scale if ok_f else np.nan
        a_sum = before["act_value"].iloc[-n:].sum() if ok_a else np.nan
        if ok_f and ok_a and a_sum > 0:
            g = (f_sum / a_sum - 1) * 100
            print(f"  {label}: 예측합 {f_sum:,.0f} / 실적합 {a_sum:,.0f} "
                  f"→ 성장률 {g:+.1f}%  ✅")
        else:
            why = []
            if not ok_f: why.append(f"예측분기 {len(fc)}<{n}")
            if not ok_a: why.append(f"예측이전 실적분기 {len(before)}<{n}")
            if ok_f and ok_a and a_sum <= 0: why.append("실적합≤0")
            print(f"  {label}: NaN  ← ★원인: {', '.join(why)}")
    print("=" * 78)


In [6]:
# ── Cell 5 · FCFF DCF 순위 추출 ─────────────────────────────────

def get_fcff_ranking(n: int = 50,
                     rank_range: Optional[Tuple[int, int]] = None,
                     dates: Union[None, str, List[str]] = None,
                     growth_model: Optional[str] = "Ensemble",   # ★v2
                     save: bool = True,
                     file_format: str = "xlsx",
                     max_upside: Optional[float] = None) -> pd.DataFrame:
    """FCFF DCF 결과를 upside 내림차순 순위로 추출.

    Parameters
    ----------
    n          : 상위 N개 (rank_range 지정 시 무시)
    rank_range : (100, 150) 처럼 순위 구간 출력
    dates      : ["2026-06-01", "2026-06-02"] 평가일 리스트. None → 최신일.
                 여러 날 중복 종목은 최신 측정일 행만 사용.
    save       : True → C:\valuation results 에 엑셀 저장 (기본 True)
    file_format: "xlsx"(기본) | "csv"
    max_upside : upside 상한 필터(%). 예: 300 → +300% 초과 이상치 제외
    growth_model : ★v2 매출성장률 산출 모델. "Ensemble"(기본)|"SARIMA"|"ETS"|"Theta".
                   None → 성장률 칼럼 생략. Upside 칼럼 옆에 4Q/8Q 성장률 삽입.
    """
    run_dates = _resolve_dates(dates, TABLE_FCFF, "FCFF")
    ph = ", ".join(f":d{i}" for i in range(len(run_dates)))
    params = {f"d{i}": d for i, d in enumerate(run_dates)}

    # 동일 ticker 가 여러 날 측정된 경우 최신 date 행 1개만 (행 내 정합성 유지)
    sql = text(f"""
        SELECT * FROM (
            SELECT
                ticker, `date` AS measured_date,
                target_price, current_price, upside_pct,
                discount_rate AS wacc, wacc_re, wacc_rd,
                g_terminal, moat_label, eva_spread,
                enterprise_value, equity_value, net_debt,
                nwc_method, revenue_quarters, forecast_model,
                ROW_NUMBER() OVER (
                    PARTITION BY ticker ORDER BY `date` DESC, id DESC
                ) AS rn
            FROM `{TABLE_FCFF}`
            WHERE `date` IN ({ph})
              AND target_price IS NOT NULL
              AND current_price > 0
              AND upside_pct IS NOT NULL
        ) t WHERE rn = 1
    """)
    raw = pd.read_sql(sql, engine, params=params).drop(columns=["rn"])
    if raw.empty:
        log("FCFF", "조건에 맞는 결과 없음")
        return raw
    if max_upside is not None:
        n_drop = int((raw["upside_pct"] > max_upside).sum())
        raw = raw[raw["upside_pct"] <= max_upside]
        if n_drop:
            log("FCFF", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_pct", ascending=False)

    out = pd.DataFrame({
        "ticker":        raw["ticker"],
        "측정일":         raw["measured_date"],
        "목표주가(원)":    raw["target_price"].round(0),
        "현재가(원)":      raw["current_price"].round(0),
        "Upside(%)":     raw["upside_pct"].round(1),
        "WACC(%)":       (raw["wacc"] * 100).round(2),
        "Re(%)":         (raw["wacc_re"] * 100).round(2),
        "Rd(%)":         (raw["wacc_rd"] * 100).round(2),
        "g_term(%)":     (raw["g_terminal"] * 100).round(2),
        "Moat":          raw["moat_label"],
        "EVA스프레드":     raw["eva_spread"].round(4),
        "기업가치EV(억원)": (raw["enterprise_value"] / 1e8).round(0),
        "순부채(억원)":    (raw["net_debt"] / 1e8).round(0),
        "NWC정의":        raw["nwc_method"],
        "매출분기수":      raw["revenue_quarters"],
        "예측모델":        raw["forecast_model"],
    })
    out = _attach_name(out)
    out = _slice_rank(out, n, rank_range)
    out = _merge_growth(out, growth_model)   # ★v2: Upside 옆 매출성장률

    print(f"\n{'='*90}")
    print(f"  [FCFF DCF] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*90}")
    display(out)
    _save_output(out, "FCFF", run_dates, save=save, file_format=file_format)
    return out


In [7]:
# ── Cell 6 · RIM 순위 추출 ──────────────────────────────────────

def get_rim_ranking(n: int = 50,
                    rank_range: Optional[Tuple[int, int]] = None,
                    dates: Union[None, str, List[str]] = None,
                    growth_model: Optional[str] = "Ensemble",   # ★v2
                    save: bool = True,
                    file_format: str = "xlsx",
                    max_upside: Optional[float] = None) -> pd.DataFrame:
    """RIM(잔여이익모형) 결과를 upside 내림차순 순위로 추출.

    파라미터는 get_fcff_ranking 과 동일.
    bv_source='IC_fallback' (음수 BV → IC 대체) 종목은 신뢰도 주의 컬럼으로 표시.
    """
    run_dates = _resolve_dates(dates, TABLE_RIM, "RIM")
    ph = ", ".join(f":d{i}" for i in range(len(run_dates)))
    params = {f"d{i}": d for i, d in enumerate(run_dates)}

    # RIM 은 (ticker, date, year_label) 구조 → 종목·날짜별 다행.
    # TP/upside 등 요약값은 행마다 동일하게 저장되므로 최신 date 의 1행만 취함.
    sql = text(f"""
        SELECT * FROM (
            SELECT
                ticker, `date` AS measured_date,
                target_price, current_price, upside_pct,
                intrinsic_value, re, g_terminal, rho, n_phase2,
                moat_label, bv_source, beta_blume,
                revenue_quarters, forecast_model,
                ROW_NUMBER() OVER (
                    PARTITION BY ticker ORDER BY `date` DESC, id DESC
                ) AS rn
            FROM `{TABLE_RIM}`
            WHERE `date` IN ({ph})
              AND target_price IS NOT NULL
              AND current_price > 0
              AND upside_pct IS NOT NULL
        ) t WHERE rn = 1
    """)
    raw = pd.read_sql(sql, engine, params=params).drop(columns=["rn"])
    if raw.empty:
        log("RIM", "조건에 맞는 결과 없음")
        return raw
    if max_upside is not None:
        n_drop = int((raw["upside_pct"] > max_upside).sum())
        raw = raw[raw["upside_pct"] <= max_upside]
        if n_drop:
            log("RIM", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_pct", ascending=False)

    out = pd.DataFrame({
        "ticker":          raw["ticker"],
        "측정일":           raw["measured_date"],
        "목표주가(원)":      raw["target_price"].round(0),
        "현재가(원)":        raw["current_price"].round(0),
        "Upside(%)":       raw["upside_pct"].round(1),
        "내재가치(억원)":     (raw["intrinsic_value"] / 1e8).round(0),
        "Re(%)":           (raw["re"] * 100).round(2),
        "g_term(%)":       (raw["g_terminal"] * 100).round(2),
        "RI지속계수ω":      raw["rho"].round(3),
        "Phase2연수":       raw["n_phase2"],
        "Moat":            raw["moat_label"],
        "β_Blume":         raw["beta_blume"].round(3),
        "BV출처":           raw["bv_source"],      # IC_fallback = 신뢰도 주의
        "매출분기수":        raw["revenue_quarters"],
        "예측모델":          raw["forecast_model"],
    })
    out = _attach_name(out)
    out = _slice_rank(out, n, rank_range)
    out = _merge_growth(out, growth_model)   # ★v2: Upside 옆 매출성장률

    print(f"\n{'='*90}")
    print(f"  [RIM] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*90}")
    display(out)
    _save_output(out, "RIM", run_dates, save=save, file_format=file_format)
    return out


In [8]:
# ── Cell 7 · Relative Valuation 순위 추출 (PER/PBR/PSR 선택) ────

def get_relative_ranking(n: int = 50,
                         rank_range: Optional[Tuple[int, int]] = None,
                         dates: Union[None, str, List[str]] = None,
                         metrics: Union[str, List[str]] = "ALL",
                         growth_model: Optional[str] = "Ensemble",   # ★v2
                         save: bool = True,
                         file_format: str = "xlsx",
                         include_excluded: bool = False,
                         max_upside: Optional[float] = None) -> pd.DataFrame:
    """상대가치 결과를 upside 내림차순 순위로 추출.

    Parameters
    ----------
    metrics : "PER" | "PBR" | "PSR" | ["PER","PBR"] | "ALL"
        ※ DB 에는 PER/PBR/PSR 3종이 저장되어 있습니다 (PCR 미산출).
        - 1개 지정 → 해당 지표 upside 기준 정렬
        - 2개 이상/ALL → 선택 지표 upside 평균 기준 정렬
    include_excluded : True → 금융 등 제외 섹터(is_excluded=1)도 포함
    나머지 파라미터는 get_fcff_ranking 과 동일.
    """
    # ── 지표 정규화 ──
    if isinstance(metrics, str):
        metrics = REL_METRICS_ALL if metrics.strip().upper() == "ALL" else [metrics]
    mets = [m.strip().upper() for m in metrics]
    bad = [m for m in mets if m not in REL_METRICS_ALL]
    if bad:
        raise ValueError(
            f"지원하지 않는 지표 {bad}. 사용 가능: {REL_METRICS_ALL} "
            f"(PCR 은 상대가치 노트북에서 산출하지 않아 DB에 없습니다 — PSR 사용 권장)")

    run_dates = _resolve_dates(dates, TABLE_REL, "Relative")
    ph = ", ".join(f":d{i}" for i in range(len(run_dates)))
    params = {f"d{i}": d for i, d in enumerate(run_dates)}
    excl_clause = "" if include_excluded else "AND is_excluded = 0"

    sql = text(f"""
        SELECT * FROM (
            SELECT
                ticker, `date` AS measured_date, sector, is_excluded,
                current_price,
                tp_per, tp_pbr, tp_psr, tp_avg,
                upside_per, upside_pbr, upside_psr, upside_avg,
                per_theory, pbr_theory, psr_theory,
                actual_per, actual_pbr, actual_psr,
                roe_y2, re_mid, g_est, eps_y2, bps_y2, npm_r2,
                ROW_NUMBER() OVER (
                    PARTITION BY ticker ORDER BY `date` DESC, id DESC
                ) AS rn
            FROM `{TABLE_REL}`
            WHERE `date` IN ({ph})
              AND current_price > 0
              {excl_clause}
        ) t WHERE rn = 1
    """)
    raw = pd.read_sql(sql, engine, params=params).drop(columns=["rn"])
    if raw.empty:
        log("REL", "조건에 맞는 결과 없음")
        return raw

    # ── 정렬 기준: 선택 지표 upside (1개=그 지표, 복수=평균) ──
    up_cols = [f"upside_{m.lower()}" for m in mets]
    raw["upside_sel"] = raw[up_cols].mean(axis=1, skipna=True)
    raw = raw[raw["upside_sel"].notna()]
    if max_upside is not None:
        n_drop = int((raw["upside_sel"] > max_upside).sum())
        raw = raw[raw["upside_sel"] <= max_upside]
        if n_drop:
            log("REL", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_sel", ascending=False)

    sel_label = "+".join(mets)
    cols = {
        "ticker":      raw["ticker"],
        "측정일":       raw["measured_date"],
        "섹터":         raw["sector"],
        "현재가(원)":    raw["current_price"].round(0),
        f"Upside_{sel_label}(%)": raw["upside_sel"].round(1),
    }
    for m in mets:  # 지표별 적정가/upside/이론·실제 멀티플
        lm = m.lower()
        cols[f"TP_{m}(원)"]    = raw[f"tp_{lm}"].round(0)
        cols[f"Upside_{m}(%)"] = raw[f"upside_{lm}"].round(1)
        cols[f"{m}_이론"]      = raw[f"{lm}_theory"].round(2)
        cols[f"{m}_실제"]      = raw[f"actual_{lm}"].round(2)
    cols.update({
        "ROE_y2(%)":  (raw["roe_y2"] * 100).round(2),
        "Re_mid(%)":  (raw["re_mid"] * 100).round(2),
        "g_est(%)":   (raw["g_est"] * 100).round(2),
        "EPS_y2(원)":  raw["eps_y2"].round(0),
        "BPS_y2(원)":  raw["bps_y2"].round(0),
        "NPM_R2":     raw["npm_r2"].round(3),
    })
    if include_excluded:
        cols["제외섹터"] = raw["is_excluded"]
    out = pd.DataFrame(cols)
    out = _attach_name(out)
    out = _slice_rank(out, n, rank_range)
    out = _merge_growth(out, growth_model)   # ★v2: Upside 옆 매출성장률

    print(f"\n{'='*90}")
    print(f"  [Relative {sel_label}] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*90}")
    display(out)
    _save_output(out, f"Relative_{sel_label}", run_dates,
                 save=save, file_format=file_format)
    return out


In [9]:
# ── Cell 8 · 3개 모형 통합 비교 순위 ────────────────────────────

def _fetch_model_upside(table: str, model_name: str,
                        dates) -> pd.DataFrame:
    """모형별 (ticker → 최신 측정일 TP/현재가/upside) 공통 추출."""
    run_dates = _resolve_dates(dates, table, model_name)
    ph = ", ".join(f":d{i}" for i in range(len(run_dates)))
    params = {f"d{i}": d for i, d in enumerate(run_dates)}

    if table == TABLE_REL:
        tp_expr, up_expr = "tp_avg", "upside_avg"
        extra_filter = "AND is_excluded = 0 AND tp_avg IS NOT NULL"
    else:
        tp_expr, up_expr = "target_price", "upside_pct"
        extra_filter = "AND target_price IS NOT NULL AND upside_pct IS NOT NULL"

    sql = text(f"""
        SELECT ticker, measured_date, tp, cp, upside FROM (
            SELECT ticker, `date` AS measured_date,
                   {tp_expr} AS tp, current_price AS cp, {up_expr} AS upside,
                   ROW_NUMBER() OVER (
                       PARTITION BY ticker ORDER BY `date` DESC, id DESC
                   ) AS rn
            FROM `{table}`
            WHERE `date` IN ({ph})
              AND current_price > 0
              {extra_filter}
        ) t WHERE rn = 1
    """)
    df = pd.read_sql(sql, engine, params=params)
    df["measured_date"] = pd.to_datetime(df["measured_date"]).dt.strftime("%Y-%m-%d")
    df.attrs["run_dates"] = run_dates
    return df


def get_combined_ranking(n: int = 50,
                         rank_range: Optional[Tuple[int, int]] = None,
                         dates: Union[None, str, List[str]] = None,
                         growth_model: Optional[str] = "Ensemble",   # ★v2
                         save: bool = True,
                         file_format: str = "xlsx",
                         min_models: int = 1,
                         max_upside: Optional[float] = None) -> pd.DataFrame:
    """FCFF · RIM · Relative(평균) 3개 모형 upside 를 종목별로 병합한 통합 순위.

    - 정렬 기준: 가용 모형 upside 의 단순평균 (Upside_평균)
    - min_models : 최소 몇 개 모형에서 평가된 종목만 포함 (기본 1, 보수적으로 보려면 3)
    - dates=None → 모형별 각각의 최신 평가일 사용
      (상대가치는 제외섹터 비포함 · tp_avg 기준)
    """
    parts = {}
    for label, tbl in [("FCFF", TABLE_FCFF), ("RIM", TABLE_RIM),
                       ("REL", TABLE_REL)]:
        try:
            mname = {"FCFF": "FCFF", "RIM": "RIM", "REL": "Relative"}[label]
            parts[label] = _fetch_model_upside(tbl, mname, dates)
        except ValueError as e:
            log("COMBINED", f"[WARN] {label} 건너뜀: {e}")

    if not parts:
        log("COMBINED", "사용 가능한 모형 결과가 없습니다.")
        return pd.DataFrame()

    merged = None
    all_run_dates = []
    for label, df in parts.items():
        all_run_dates += df.attrs.get("run_dates", [])
        sub = df.rename(columns={
            "tp": f"TP_{label}(원)", "upside": f"Upside_{label}(%)",
            "cp": f"_cp_{label}", "measured_date": f"_dt_{label}"})
        merged = sub if merged is None else merged.merge(sub, on="ticker", how="outer")

    up_cols = [c for c in merged.columns if c.startswith("Upside_")]
    cp_cols = [c for c in merged.columns if c.startswith("_cp_")]
    dt_cols = [c for c in merged.columns if c.startswith("_dt_")]

    merged["평가모형수"] = merged[up_cols].notna().sum(axis=1)
    merged = merged[merged["평가모형수"] >= int(min_models)]
    merged["Upside_평균(%)"] = merged[up_cols].mean(axis=1, skipna=True)
    if max_upside is not None:
        merged = merged[merged["Upside_평균(%)"] <= max_upside]
    merged["현재가(원)"] = merged[cp_cols].bfill(axis=1).iloc[:, 0]
    merged["최신측정일"] = merged[dt_cols].max(axis=1)

    merged = merged.sort_values("Upside_평균(%)", ascending=False)

    ordered = (["ticker", "최신측정일", "현재가(원)", "Upside_평균(%)", "평가모형수"]
               + sorted(up_cols)
               + sorted(c for c in merged.columns if c.startswith("TP_")))
    out = merged[ordered].copy()
    out["현재가(원)"] = out["현재가(원)"].round(0)
    for c in out.columns:
        if c.startswith("Upside"):
            out[c] = out[c].round(1)
        elif c.startswith("TP_"):
            out[c] = out[c].round(0)
    out = _attach_name(out)
    out = _slice_rank(out, n, rank_range)
    out = _merge_growth(out, growth_model)   # ★v2: Upside 옆 매출성장률

    run_dates = sorted(set(all_run_dates))
    print(f"\n{'='*100}")
    print(f"  [통합 Combined] FCFF·RIM·Relative upside 평균 순위  |  "
          f"평가일 {run_dates}  |  min_models={min_models}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*100}")
    display(out)
    _save_output(out, "Combined", run_dates, save=save, file_format=file_format)
    return out


In [10]:
# ── Cell 9-B (★v5) · 추출된 순위표 병합 → 통합 upside 분석 ──────
#
#  get_fcff_ranking / get_rim_ranking / get_relative_ranking 이 "반환한"
#  DataFrame 2~3개를 ticker 기준으로 merge 하여
#    1) 모형별 Upside 칼럼 + 동일가중 평균 Upside_평균(%) 산출
#    2) 매출성장률_4Q/8Q(%) 유지 (모형 간 coalesce — 먼저 입력된 df 우선)
#    3) Moat 등급 표시 (FCFF/RIM 에서 coalesce — Relative 에는 Moat 없음)
#    4) Upside_평균(%) 내림차순 정렬
#  하는 함수입니다. DB 재조회 없이 이미 추출된 결과만 사용합니다.
#
#  ※ get_combined_ranking 과의 차이: combined 는 DB에서 다시 읽지만,
#    이 함수는 화면에 띄워 둔 순위표(필터·rank_range 적용 결과)를 그대로 병합합니다.

_MERGE_G_MAP = {"매출성장률_4Q(%)": "_g4", "매출성장률_8Q(%)": "_g8"}


def _detect_model_label(df: pd.DataFrame) -> str:
    """순위표 DataFrame 의 출처 모형 자동 판별 → 'FCFF' / 'RIM' / 'REL'."""
    cols = set(map(str, df.columns))
    if {"WACC(%)", "NWC정의", "기업가치EV(억원)"} & cols:
        return "FCFF"
    if {"RI지속계수ω", "BV출처", "내재가치(억원)"} & cols:
        return "RIM"
    if "섹터" in cols and any(c.startswith("Upside_") for c in cols):
        return "REL"
    raise ValueError(
        "모형 판별 불가 — get_fcff_ranking / get_rim_ranking / "
        "get_relative_ranking 이 반환한 DataFrame 을 그대로 입력하세요. "
        f"(columns 앞부분: {list(df.columns)[:8]})")


def _pick_upside_col(df: pd.DataFrame, label: str) -> str:
    """모형별 대표 upside 칼럼명 결정.

    FCFF/RIM → 'Upside(%)'
    REL      → 집계 upside (예: 'Upside_PER+PBR+PSR(%)') = 첫 'Upside_' 칼럼
               (지표 1개로 추출한 경우 'Upside_PER(%)' 처럼 그 지표 자체)
    """
    if label in ("FCFF", "RIM"):
        if "Upside(%)" in df.columns:
            return "Upside(%)"
    else:
        for c in df.columns:
            s = str(c)
            if s.startswith("Upside_") and s.endswith("(%)"):
                return s
    raise ValueError(f"[{label}] upside 칼럼을 찾을 수 없습니다. "
                     f"(columns 앞부분: {list(df.columns)[:8]})")


def merge_upside_rankings(dfs: Union[List[pd.DataFrame], Dict[str, pd.DataFrame]],
                          n: Optional[int] = None,
                          rank_range: Optional[Tuple[int, int]] = None,
                          min_models: int = 1,
                          max_upside: Optional[float] = None,
                          save: bool = True,
                          file_format: str = "xlsx") -> pd.DataFrame:
    """추출된 순위표 2~3개를 ticker 기준 병합한 통합 upside 순위.   ★v5

    Parameters
    ----------
    dfs : list 또는 dict (2~3개)
        get_fcff_ranking / get_rim_ranking / get_relative_ranking 반환 DataFrame.
        - list → 칼럼 구성으로 모형 자동 판별
        - dict → {"FCFF": df1, "RIM": df2, "REL": df3} 라벨 직접 지정
    n / rank_range : 출력 구간. 기본 None → 전체 출력.
    min_models : 최소 평가 모형 수 (기본 1 = 합집합).
                 입력 df 개수와 같게 주면 모든 모형에서 평가된 교집합만.
    max_upside : Upside_평균(%) 상한 필터 (이상치 제외용, 예: 300)
    save : True → OUTPUT_DIR 에 'Merged_{모형들}_valuation_...' 저장

    Returns
    -------
    rank · ticker · 종목명 · 최신측정일 · Moat · Upside_평균(%) · 평가모형수 ·
    모형별 Upside(%) · 매출성장률_4Q/8Q(%)  — Upside_평균 내림차순.
    평균은 해당 종목이 평가된 모형들의 **동일가중 단순평균** (결측 모형 제외).
    """
    # ── 1) 입력 정규화 + 모형 판별 ──────────────────────────────
    if isinstance(dfs, dict):
        items = [(str(k).strip().upper(), v) for k, v in dfs.items()]
    else:
        items = [(_detect_model_label(df), df) for df in dfs]
    if not 2 <= len(items) <= 3:
        raise ValueError(f"DataFrame 은 2~3개 입력하세요 (현재 {len(items)}개).")
    labels = [lb for lb, _ in items]
    if len(set(labels)) != len(labels):
        raise ValueError(f"같은 모형이 중복 입력되었습니다: {labels}")

    # ── 2) 모형별 핵심 칼럼 추출 → ticker 기준 outer merge ──────
    merged: Optional[pd.DataFrame] = None
    run_dates: set = set()
    for label, df in items:
        if df is None or df.empty:
            raise ValueError(f"[{label}] 입력 DataFrame 이 비어 있습니다.")
        if "ticker" not in df.columns:
            raise ValueError(f"[{label}] 'ticker' 칼럼이 없습니다.")
        d = df.copy()
        up_col = _pick_upside_col(d, label)

        d["_tk"] = d["ticker"].map(_norm_tk)
        n_dup = int(d["_tk"].duplicated().sum())
        if n_dup:
            log("MERGE", f"⚠️ [{label}] ticker 중복 {n_dup}건 → 첫 행(상위 순위) 사용")
        d = d.drop_duplicates("_tk", keep="first")

        part = pd.DataFrame({"_tk": d["_tk"].values})
        part[f"Upside_{label}(%)"] = pd.to_numeric(d[up_col], errors="coerce").values
        if "종목명" in d.columns:
            part[f"_nm_{label}"] = d["종목명"].values
        if "Moat" in d.columns:
            part[f"_moat_{label}"] = d["Moat"].values
        for gcol, tag in _MERGE_G_MAP.items():
            if gcol in d.columns:
                part[f"{tag}_{label}"] = pd.to_numeric(d[gcol], errors="coerce").values
        if "측정일" in d.columns:
            part[f"_dt_{label}"] = d["측정일"].astype(str).values
            run_dates |= set(part[f"_dt_{label}"].dropna())

        log("MERGE", f"[{label}] {len(part):,}종목 | upside='{up_col}' | "
                     f"Moat={'O' if 'Moat' in d.columns else 'X'} | "
                     f"성장률={'O' if _GROWTH_COLS[0] in d.columns else 'X'}")
        merged = part if merged is None else merged.merge(part, on="_tk", how="outer")

    # ── 3) coalesce (먼저 입력된 df 우선) ───────────────────────
    def _coalesce(prefix: str) -> pd.Series:
        cols = [f"{prefix}_{lb}" for lb in labels if f"{prefix}_{lb}" in merged.columns]
        if not cols:
            return pd.Series(np.nan, index=merged.index)
        s = merged[cols[0]]
        for c in cols[1:]:
            s = s.combine_first(merged[c])
        return s

    up_cols = [f"Upside_{lb}(%)" for lb in labels]
    out = pd.DataFrame({"ticker": merged["_tk"]})
    out["종목명"]   = _coalesce("_nm").fillna("")
    dt_cols = [c for c in merged.columns if c.startswith("_dt_")]
    # 변경 후
    if dt_cols:   # outer merge 로 생긴 NaN(float) + 문자열 혼합 → datetime 변환 후 max
        _dt = merged[dt_cols].apply(lambda s: pd.to_datetime(s, errors="coerce"))
        out["최신측정일"] = _dt.max(axis=1).dt.strftime("%Y-%m-%d").fillna("")
    else:
        out["최신측정일"] = ""
    out["Moat"]    = _coalesce("_moat")
    for c in up_cols:
        out[c] = merged[c].round(1)
    out["평가모형수"]      = merged[up_cols].notna().sum(axis=1)
    out["Upside_평균(%)"] = merged[up_cols].mean(axis=1, skipna=True).round(1)  # 동일가중
    out["매출성장률_4Q(%)"] = _coalesce("_g4")
    out["매출성장률_8Q(%)"] = _coalesce("_g8")

    # ── 4) 필터 → 평균 upside 내림차순 정렬 ─────────────────────
    n_all = len(out)
    out = out[out["평가모형수"] >= int(min_models)]
    if max_upside is not None:
        n_drop = int((out["Upside_평균(%)"] > max_upside).sum())
        out = out[out["Upside_평균(%)"] <= max_upside]
        if n_drop:
            log("MERGE", f"Upside_평균 > {max_upside}% 이상치 {n_drop}개 제외")
    out = out.sort_values("Upside_평균(%)", ascending=False, na_position="last")

    ordered = (["ticker", "종목명", "최신측정일", "Moat",
                "Upside_평균(%)", "평가모형수"] + up_cols + _GROWTH_COLS)
    out = out[ordered]
    out = _slice_rank(out, n if n is not None else len(out), rank_range)

    # ── 5) 출력 + 저장 ──────────────────────────────────────────
    tag = "+".join(labels)
    log("MERGE", f"병합 완료: 합집합 {n_all:,}종목 → "
                 f"min_models≥{min_models} 적용 후 {len(out):,}종목 출력 "
                 f"(평균 = {tag} 동일가중)")
    print(f"\n{'='*100}")
    print(f"  [Merged {tag}] 동일가중 평균 upside 순위  |  "
          f"min_models={min_models}  |  "
          f"{'rank ' + str(rank_range) if rank_range else ('전체' if n is None else 'TOP ' + str(n))}")
    print(f"{'='*100}")
    display(out)
    rd = sorted(run_dates) if run_dates else [datetime.now().strftime("%Y-%m-%d")]
    _save_output(out, f"Merged_{tag}", rd, save=save, file_format=file_format)
    return out


In [11]:
# ── Cell 9 · 0단계: valuation 수행 날짜부터 확인 ────────────────
dates_df = get_valuation_dates("all")   # "fcff" / "rim" / "relative" 개별 조회 가능
display(dates_df)


,model,run_date,n_tickers,n_rows
0,FCFF,2026-06-01,1266,10128
1,RIM,2026-06-12,1272,18890
2,RIM,2026-06-06,1272,18862
3,Relative,2026-06-06,1277,1277
4,Relative,2026-04-26,7,7


## 사용 예시

```python
# 1) FCFF 상위 30개, 최신 평가일, 엑셀 저장(기본 True)
fcff_top = get_fcff_ranking(n=30)

# 2) RIM 100~150위 구간, 특정 날짜 2일 (중복 종목은 최신 측정일 우선)
rim_mid = get_rim_ranking(rank_range=(100, 150),
                          dates=["2026-06-01", "2026-06-02"])

# 3) 상대가치 — PER 단독 / PBR+PSR 조합 / 전체
rel_per = get_relative_ranking(n=50, metrics="PER")
rel_mix = get_relative_ranking(n=50, metrics=["PBR", "PSR"])
rel_all = get_relative_ranking(n=50, metrics="ALL", save=False)   # 저장 생략

# 4) 통합 — 3개 모형 모두 평가된 종목만, upside 평균 상위 50
combo = get_combined_ranking(n=50, min_models=3)

# 5) ★v2 매출 성장률 — 순위표에 자동 병합 (기본 Ensemble)
fcff = get_fcff_ranking(n=30)                          # Ensemble 성장률 포함
fcff_s = get_fcff_ranking(n=30, growth_model="SARIMA")  # SARIMA 성장률
fcff_x = get_fcff_ranking(n=30, growth_model=None)      # 성장률 칼럼 생략

# 6) ★v3 성장률 NaN 원인 진단 / 테이블 직접 확인
diagnose_growth("005930")                  # 단계별 원인 출력
tbl = build_growth_table("Ensemble")        # 전 종목 성장률 테이블 (캐시)
tbl = build_growth_table("Ensemble", refresh=True)  # 예측 재실행 후 갱신
# ★v4: 단위 불일치는 자동 보정(unit_adjust='auto'). 끄려면:
# tbl = build_growth_table("Ensemble", refresh=True, unit_adjust=1.0)

# 7) 순수 매출 성장률 단독 조회
g_all  = get_revenue_growth()                            # 전체 종목, Ensemble
g_one  = get_revenue_growth("005930", model="Theta", detail=True)
g_list = get_revenue_growth(["A005930", "A000660"], model="ETS")

# 8-2) ★v5 추출된 순위표 병합 — 동일가중 평균 upside
fcff_range = get_fcff_ranking(rank_range=(0, 1277), dates=["2026-06-01"])
rel_top    = get_relative_ranking(rank_range=(0, 1277), dates=["2026-06-06"], metrics="ALL")
rim_top    = get_rim_ranking(rank_range=(0, 1277), dates=["2026-06-06"])

merged = merge_upside_rankings([fcff_range, rel_top, rim_top])        # 3개 자동 판별, 합집합
merged = merge_upside_rankings([fcff_range, rim_top])                  # 2개만도 가능
merged = merge_upside_rankings([fcff_range, rel_top, rim_top],
                               min_models=3)                           # 3개 모형 모두 평가된 교집합만
merged = merge_upside_rankings({"FCFF": fcff_range, "REL": rel_top},   # 라벨 직접 지정
                               max_upside=300, save=False)

# 8) 데이터 품질 필터 — upside +300% 초과 이상치 제외
fcff_clean = get_fcff_ranking(n=100, max_upside=300)
```

저장 파일명 형식: `C:\valuation results\FCFF_valuation_2026-06-01_to_2026-06-02_2026-06-05.xlsx`
(= `{모형}_valuation_{수행날짜}_{출력날짜}.xlsx`)


In [20]:
# ── Cell 10 · 실행 예시 (필요 시 주석 해제) ─────────────────────
# fcff_top = get_fcff_ranking(n=30)   # ★v2: Upside 옆에 매출성장률_4Q/8Q 자동 포함
# g_df   = get_revenue_growth(["005930"], model="Ensemble", detail=True)
# rim_top  = get_rim_ranking(n=30)
# rel_top  = get_relative_ranking(n=30, metrics="ALL")
# combo    = get_combined_ranking(n=30, min_models=2)
fcff_range = get_fcff_ranking(rank_range=(0, 1277), dates=["2026-06-01"])

[21:20:28][GROWTH] 순위표 1266종목 중 예측DB 매칭 1266종목 / 성장률 유효 1176종목 (model=Ensemble)

  [FCFF DCF] upside 순위  |  평가일 ['2026-06-01']  |  rank (0, 1277)


,rank,ticker,종목명,측정일,목표주가(원),현재가(원),Upside(%),매출성장률_4Q(%),매출성장률_8Q(%),WACC(%),Re(%),Rd(%),g_term(%),Moat,EVA스프레드,기업가치EV(억원),순부채(억원),NWC정의,매출분기수,예측모델
0,1,A161390,,2026-06-01,2.397774e+157,63000.0,3.805990e+154,31.1,91.5,10.41,10.41,0.66,0.9,No moat,-0.0163,2.925014e+157,0.0,legacy_proxy,55,Ensemble
1,2,A054540,,2026-06-01,1.293598e+07,10000.0,1.292598e+05,5.4,8.4,11.09,11.09,0.89,4.0,No moat,-0.0831,1.681677e+06,0.0,legacy_proxy,26,Ensemble
2,3,A263750,,2026-06-01,5.581133e+06,43900.0,1.261330e+04,NaN,NaN,8.98,8.98,0.86,4.0,No moat,0.1426,3.427899e+06,0.0,legacy_proxy,26,Ensemble
3,4,A005710,,2026-06-01,4.956660e+05,11100.0,4.365500e+03,4.2,23.0,9.22,9.22,0.57,4.0,Wide moat,0.1725,9.932000e+04,0.0,legacy_proxy,26,Ensemble
4,5,A032190,,2026-06-01,8.355710e+05,20650.0,3.946400e+03,5.6,32.5,11.48,11.48,0.10,4.0,No moat,-0.0776,3.200240e+05,0.0,legacy_proxy,26,Ensemble
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1261,1262,A038110,,2026-06-01,-1.286540e+05,2720.0,-4.829900e+03,9.8,17.5,10.43,10.43,1.13,4.0,No moat,-0.0907,-5.296600e+04,0.0,legacy_proxy,26,Ensemble
1262,1263,A002390,,2026-06-01,-3.571050e+06,8470.0,-4.226120e+04,5.5,6.6,9.60,9.60,0.89,4.0,No moat,-0.0891,-4.639130e+05,0.0,legacy_proxy,66,Ensemble
1263,1264,A052400,,2026-06-01,-6.112647e+08,57300.0,-1.066880e+06,-182.4,-393.7,9.53,9.53,0.99,0.9,Wide moat,0.9073,-8.784627e+07,0.0,legacy_proxy,26,Ensemble
1264,1265,A098460,,2026-06-01,-2.208743e+96,35500.0,-6.221812e+93,9.5,11.7,11.59,11.59,1.61,0.9,No moat,-0.0201,-1.454371e+96,0.0,legacy_proxy,26,Ensemble


[21:20:28][SAVE] 저장 완료: C:\valuation results\FCFF_valuation_2026-06-01_2026-06-06.xlsx  (1266행)


In [21]:
rel_top  = get_relative_ranking(rank_range=(0, 1277), dates=["2026-06-06"], metrics="ALL")
rel_top

[21:20:28][GROWTH] 순위표 1264종목 중 예측DB 매칭 1264종목 / 성장률 유효 1176종목 (model=Ensemble)

  [Relative PER+PBR+PSR] upside 순위  |  평가일 ['2026-06-06']  |  rank (0, 1277)


,rank,ticker,종목명,측정일,섹터,현재가(원),Upside_PER+PBR+PSR(%),매출성장률_4Q(%),매출성장률_8Q(%),TP_PER(원),...,TP_PSR(원),Upside_PSR(%),PSR_이론,PSR_실제,ROE_y2(%),Re_mid(%),g_est(%),EPS_y2(원),BPS_y2(원),NPM_R2
0,1,A263750,,2026-06-06,Unknown,40750.0,205519.8,NaN,NaN,94737559.0,...,94737559.0,232384.8,22.67,0.01,32.58,9.93,8.0,3047486.0,6110630.0,0.977
1,2,A054540,,2026-06-06,Unknown,8750.0,30419.7,5.4,8.4,3801930.0,...,3801930.0,43350.6,0.14,0.00,5.31,11.15,4.0,1100699.0,2220892.0,0.363
2,3,A298690,,2026-06-06,Unknown,1655.0,15348.0,NaN,NaN,164055.0,...,164055.0,9812.7,11.37,0.11,126.22,10.45,8.0,4572.0,9690.0,0.833
3,4,A036710,,2026-06-06,Unknown,4770.0,13949.8,10.1,19.7,460069.0,...,460069.0,9545.1,10.56,0.11,102.81,11.49,8.0,14418.0,33236.0,0.404
4,5,A002390,,2026-06-06,Unknown,8090.0,12966.7,5.5,6.6,1528064.0,...,1528064.0,18788.3,0.07,0.00,3.70,10.27,3.7,565483.0,1151589.0,-1.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1259,1260,A028300,,2026-06-06,Unknown,50700.0,-99.8,32.8,65.4,NaN,...,NaN,NaN,NaN,52.24,-17.30,10.96,0.0,-1281.0,1154.0,-1.000
1260,1261,A087010,,2026-06-06,Unknown,258000.0,-99.9,0.9,50.7,NaN,...,NaN,NaN,NaN,771.06,-33.27,10.97,0.0,-1332.0,3186.0,-1.000
1261,1262,A950160,,2026-06-06,Unknown,101400.0,-99.9,-99.9,-99.9,NaN,...,NaN,NaN,NaN,1792.63,-40.68,9.56,0.0,-406.0,668.0,-1.000
1262,1263,A140410,,2026-06-06,Unknown,63200.0,-100.0,6.2,-37.7,NaN,...,NaN,NaN,NaN,246.78,-22.17,10.84,0.0,-119.0,273.0,-1.000


[21:20:29][SAVE] 저장 완료: C:\valuation results\Relative_PER+PBR+PSR_valuation_2026-06-06_2026-06-06.xlsx  (1264행)


,rank,ticker,종목명,측정일,섹터,현재가(원),Upside_PER+PBR+PSR(%),매출성장률_4Q(%),매출성장률_8Q(%),TP_PER(원),...,TP_PSR(원),Upside_PSR(%),PSR_이론,PSR_실제,ROE_y2(%),Re_mid(%),g_est(%),EPS_y2(원),BPS_y2(원),NPM_R2
0,1,A263750,,2026-06-06,Unknown,40750.0,205519.8,NaN,NaN,94737559.0,...,94737559.0,232384.8,22.67,0.01,32.58,9.93,8.0,3047486.0,6110630.0,0.977
1,2,A054540,,2026-06-06,Unknown,8750.0,30419.7,5.4,8.4,3801930.0,...,3801930.0,43350.6,0.14,0.00,5.31,11.15,4.0,1100699.0,2220892.0,0.363
2,3,A298690,,2026-06-06,Unknown,1655.0,15348.0,NaN,NaN,164055.0,...,164055.0,9812.7,11.37,0.11,126.22,10.45,8.0,4572.0,9690.0,0.833
3,4,A036710,,2026-06-06,Unknown,4770.0,13949.8,10.1,19.7,460069.0,...,460069.0,9545.1,10.56,0.11,102.81,11.49,8.0,14418.0,33236.0,0.404
4,5,A002390,,2026-06-06,Unknown,8090.0,12966.7,5.5,6.6,1528064.0,...,1528064.0,18788.3,0.07,0.00,3.70,10.27,3.7,565483.0,1151589.0,-1.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1259,1260,A028300,,2026-06-06,Unknown,50700.0,-99.8,32.8,65.4,NaN,...,NaN,NaN,NaN,52.24,-17.30,10.96,0.0,-1281.0,1154.0,-1.000
1260,1261,A087010,,2026-06-06,Unknown,258000.0,-99.9,0.9,50.7,NaN,...,NaN,NaN,NaN,771.06,-33.27,10.97,0.0,-1332.0,3186.0,-1.000
1261,1262,A950160,,2026-06-06,Unknown,101400.0,-99.9,-99.9,-99.9,NaN,...,NaN,NaN,NaN,1792.63,-40.68,9.56,0.0,-406.0,668.0,-1.000
1262,1263,A140410,,2026-06-06,Unknown,63200.0,-100.0,6.2,-37.7,NaN,...,NaN,NaN,NaN,246.78,-22.17,10.84,0.0,-119.0,273.0,-1.000


In [22]:
rim_top  = get_rim_ranking(rank_range=(0, 1277), dates=["2026-06-06"])
rim_top

[21:20:30][GROWTH] 순위표 1272종목 중 예측DB 매칭 1272종목 / 성장률 유효 1182종목 (model=Ensemble)

  [RIM] upside 순위  |  평가일 ['2026-06-06']  |  rank (0, 1277)


,rank,ticker,종목명,측정일,목표주가(원),현재가(원),Upside(%),매출성장률_4Q(%),매출성장률_8Q(%),내재가치(억원),Re(%),g_term(%),RI지속계수ω,Phase2연수,Moat,β_Blume,BV출처,매출분기수,예측모델
0,1,A005110,,2026-06-06,1613189.0,1254.0,128543.5,NaN,-264.4,330436.0,9.58,8.58,0.98,28,Wide moat,0.761,Equity,66,Ensemble
1,2,A272450,,2026-06-06,1303517.0,5700.0,22768.7,NaN,NaN,671750.0,10.44,9.44,0.98,28,Wide moat,0.883,Equity,33,Ensemble
2,3,A040300,,2026-06-06,65130.0,2415.0,2596.9,-5.6,NaN,31052.0,9.07,8.07,0.88,10,No moat,0.688,Equity,26,Ensemble
3,4,A950130,,2026-06-06,71765.0,2725.0,2533.6,-99.8,-99.8,24047.0,9.00,8.00,0.88,10,No moat,0.678,Equity,26,Ensemble
4,5,A298690,,2026-06-06,40420.0,1655.0,2342.3,NaN,NaN,47123.0,9.92,8.92,0.88,10,No moat,0.810,Equity,31,Ensemble
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267,1268,A298000,,2026-06-06,-11716.0,38900.0,-130.1,-7.2,-17.8,-441.0,10.32,9.32,0.88,10,No moat,0.866,Equity,32,Ensemble
1268,1269,A088800,,2026-06-06,-963.0,3000.0,-132.1,-15.2,-6.8,-727.0,11.60,10.60,0.88,10,No moat,1.050,Equity,26,Ensemble
1269,1270,A062970,,2026-06-06,-894.0,2525.0,-135.4,-113.8,-137.3,-463.0,11.47,10.47,0.88,10,No moat,1.031,Equity,26,Ensemble
1270,1271,A099190,,2026-06-06,-7553.0,16800.0,-145.0,3.0,10.7,-2158.0,9.21,8.21,0.88,10,No moat,0.707,Equity,26,Ensemble


[21:20:30][SAVE] 저장 완료: C:\valuation results\RIM_valuation_2026-06-06_2026-06-06.xlsx  (1272행)


,rank,ticker,종목명,측정일,목표주가(원),현재가(원),Upside(%),매출성장률_4Q(%),매출성장률_8Q(%),내재가치(억원),Re(%),g_term(%),RI지속계수ω,Phase2연수,Moat,β_Blume,BV출처,매출분기수,예측모델
0,1,A005110,,2026-06-06,1613189.0,1254.0,128543.5,NaN,-264.4,330436.0,9.58,8.58,0.98,28,Wide moat,0.761,Equity,66,Ensemble
1,2,A272450,,2026-06-06,1303517.0,5700.0,22768.7,NaN,NaN,671750.0,10.44,9.44,0.98,28,Wide moat,0.883,Equity,33,Ensemble
2,3,A040300,,2026-06-06,65130.0,2415.0,2596.9,-5.6,NaN,31052.0,9.07,8.07,0.88,10,No moat,0.688,Equity,26,Ensemble
3,4,A950130,,2026-06-06,71765.0,2725.0,2533.6,-99.8,-99.8,24047.0,9.00,8.00,0.88,10,No moat,0.678,Equity,26,Ensemble
4,5,A298690,,2026-06-06,40420.0,1655.0,2342.3,NaN,NaN,47123.0,9.92,8.92,0.88,10,No moat,0.810,Equity,31,Ensemble
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267,1268,A298000,,2026-06-06,-11716.0,38900.0,-130.1,-7.2,-17.8,-441.0,10.32,9.32,0.88,10,No moat,0.866,Equity,32,Ensemble
1268,1269,A088800,,2026-06-06,-963.0,3000.0,-132.1,-15.2,-6.8,-727.0,11.60,10.60,0.88,10,No moat,1.050,Equity,26,Ensemble
1269,1270,A062970,,2026-06-06,-894.0,2525.0,-135.4,-113.8,-137.3,-463.0,11.47,10.47,0.88,10,No moat,1.031,Equity,26,Ensemble
1270,1271,A099190,,2026-06-06,-7553.0,16800.0,-145.0,3.0,10.7,-2158.0,9.21,8.21,0.88,10,No moat,0.707,Equity,26,Ensemble


In [23]:
# ── ★v5 실행: 위에서 추출한 3개 순위표 병합 ──────────────────
merged = merge_upside_rankings([fcff_range, rel_top, rim_top])
merged

[21:20:32][MERGE] [FCFF] 1,266종목 | upside='Upside(%)' | Moat=O | 성장률=O
[21:20:32][MERGE] [REL] 1,264종목 | upside='Upside_PER+PBR+PSR(%)' | Moat=X | 성장률=O
[21:20:32][MERGE] [RIM] 1,272종목 | upside='Upside(%)' | Moat=O | 성장률=O
[21:20:32][MERGE] 병합 완료: 합집합 1,277종목 → min_models≥1 적용 후 1,277종목 출력 (평균 = FCFF+REL+RIM 동일가중)

  [Merged FCFF+REL+RIM] 동일가중 평균 upside 순위  |  min_models=1  |  전체


,rank,ticker,종목명,최신측정일,Moat,Upside_평균(%),평가모형수,Upside_FCFF(%),Upside_REL(%),Upside_RIM(%),매출성장률_4Q(%),매출성장률_8Q(%)
0,1,A161390,,2026-06-06,No moat,1.902995e+154,2,3.805990e+154,NaN,41.9,31.1,91.5
1,2,A263750,,2026-06-06,No moat,7.273070e+04,3,1.261330e+04,205519.8,58.9,NaN,NaN
2,3,A054540,,2026-06-06,No moat,5.324940e+04,3,1.292598e+05,30419.7,68.8,5.4,8.4
3,4,A005110,,2026-06-06,No moat,4.274770e+04,3,-2.745000e+02,-25.9,128543.5,NaN,-264.4
4,5,A272450,,2026-06-06,No moat,1.138720e+04,3,2.388300e+03,9004.5,22768.7,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1272,1273,A038110,,2026-06-06,No moat,-1.505700e+03,3,-4.829900e+03,132.7,180.1,9.8,17.5
1273,1274,A002390,,2026-06-06,No moat,-9.749800e+03,3,-4.226120e+04,12966.7,45.2,5.5,6.6
1274,1275,A052400,,2026-06-06,Wide moat,-5.334106e+05,2,-1.066880e+06,NaN,58.3,-182.4,-393.7
1275,1276,A098460,,2026-06-06,No moat,-3.110906e+93,2,-6.221812e+93,NaN,-87.0,9.5,11.7


[21:20:33][SAVE] 저장 완료: C:\valuation results\Merged_FCFF+REL+RIM_valuation_2026-06-01_to_2026-06-06_2026-06-06.xlsx  (1277행)


,rank,ticker,종목명,최신측정일,Moat,Upside_평균(%),평가모형수,Upside_FCFF(%),Upside_REL(%),Upside_RIM(%),매출성장률_4Q(%),매출성장률_8Q(%)
0,1,A161390,,2026-06-06,No moat,1.902995e+154,2,3.805990e+154,NaN,41.9,31.1,91.5
1,2,A263750,,2026-06-06,No moat,7.273070e+04,3,1.261330e+04,205519.8,58.9,NaN,NaN
2,3,A054540,,2026-06-06,No moat,5.324940e+04,3,1.292598e+05,30419.7,68.8,5.4,8.4
3,4,A005110,,2026-06-06,No moat,4.274770e+04,3,-2.745000e+02,-25.9,128543.5,NaN,-264.4
4,5,A272450,,2026-06-06,No moat,1.138720e+04,3,2.388300e+03,9004.5,22768.7,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1272,1273,A038110,,2026-06-06,No moat,-1.505700e+03,3,-4.829900e+03,132.7,180.1,9.8,17.5
1273,1274,A002390,,2026-06-06,No moat,-9.749800e+03,3,-4.226120e+04,12966.7,45.2,5.5,6.6
1274,1275,A052400,,2026-06-06,Wide moat,-5.334106e+05,2,-1.066880e+06,NaN,58.3,-182.4,-393.7
1275,1276,A098460,,2026-06-06,No moat,-3.110906e+93,2,-6.221812e+93,NaN,-87.0,9.5,11.7
